# Prepare a Raiders experimental dataset

This notebook turns the original thread records into inspectable posts, explicit quote links and separate reference/development/evaluation inputs. Start with the **1,000-thread preview**, inspect the results, then set `MAX_THREADS = None` to process our full 1 GB slice.

The slice contains the first threads in archive order, dated June 30-July 12, 2016. It is not a random sample. This preparation performs no BERT inference. Full-slice HTML parsing takes longer and retains more rows in memory.

Reusable parsing functions live in `utils/raiders.py`; the research choices and tables are here. Source: [original release](https://zenodo.org/records/3606810) and [4chan field definitions](https://github.com/4chan/4chan-API/blob/master/pages/Threads.md).

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import hashlib
import html
import json
import subprocess

import pandas as pd
from IPython.display import HTML, display
from utils.anchors import anchor_terms
from utils.raiders import iter_threads, parse_post_html, prepare_raiders, thread_role, select_model_inputs

ROOT = Path.cwd()
if not (ROOT / "utils/raiders.py").is_file():
    raise RuntimeError("Open this notebook from the repository root.")
SOURCE = ROOT / "data/raiders/raiders_first_1GB.ndjson"
MAX_THREADS = 1000             # Preview; use None for the whole 1 GB slice.
SPLIT_SALT = "raiders-v1"      # Keep fixed when comparing methods.
REFERENCE_FRACTION = 0.20
DEVELOPMENT_FRACTION = 0.20   # The remaining approximately 60% is evaluation.
DISCOVERY_SCOPE = "seed_posts"  # Alternative experiment: "all_retrieved_posts".
EXPORT = True

# Provisional topic seeds, not a claim to reproduce Wu's retrieval vocabulary.
TOPIC_SEEDS = ["jew", "jews", "jewish", "judaism", "zionist", "zionists", "zionism", "antisemitism", "antisemitic"]
# Include existing anchor variants so reference contexts remain available.
SEED_TERMS = list(dict.fromkeys(TOPIC_SEEDS + [variant for variants in anchor_terms.values() for variant in variants]))
pd.set_option("display.max_colwidth", 160)
print(f"Source: {SOURCE.name} ({SOURCE.stat().st_size / 1e9:.3f} GB)")
print(f"Thread limit: {MAX_THREADS}; seed phrases: {len(SEED_TERMS)}")
display(pd.DataFrame({"retrieval_seed": SEED_TERMS}))

Source: raiders_first_1GB.ndjson (1.000 GB)
Thread limit: 1000; seed phrases: 30


,retrieval_seed
0,jew
1,jews
2,jewish
3,judaism
4,zionist
5,zionists
6,zionism
7,antisemitism
8,antisemitic
9,kike


## 1. Inspect the original structure and cleaning

One NDJSON line is one collected thread. `resto` identifies its opening post; it is not a direct parent ID. Explicit quote links are recovered from HTML before cleaning. A post may reference several posts, other boards, or a post absent from our slice.

We preserve raw HTML and subject metadata. For model text we decode HTML entities, retain line breaks and quoted language, and remove numeric quote-link labels such as `>>79300104`. We do not remove stopwords, lowercase the readable text, or run HMApp tokenization here. Subject text is retained separately and is not appended to model text.

In [2]:
preview_line, preview_thread = next(iter_threads(SOURCE, max_threads=1))
# Find one example with an explicit quote link without scanning the full file.
example = preview_thread["posts"][0]
for _, thread in iter_threads(SOURCE, max_threads=20):
    quoted = [post for post in thread["posts"] if 'class="quotelink"' in post.get("com", "")]
    if quoted:
        example = quoted[0]
        break
example_text, example_quotes = parse_post_html(example.get("com", ""))
display(pd.DataFrame([
    {"view": "Raw HTML", "value": example.get("com", "")},
    {"view": "Readable text", "value": example_text},
    {"view": "Explicit quote targets", "value": str(example_quotes)},
]))
print("Source thread keys:", list(preview_thread))
print("Opening-post fields:", list(preview_thread["posts"][0]))

Source thread keys: ['posts']
Opening-post fields: ['archived_on', 'replies', 'images', 'archived', 'no', 'fsize', 'filename', 'tim', 'closed', 'country_name', 'bumplimit', 'time', 'imagelimit', 'semantic_url', 'now', 'md5', 'name', 'tn_w', 'h', 'ext', 'resto', 'w', 'tn_h', 'country', 'com', 'entities', 'perspectives']


,view,value
0,Raw HTML,"<a href=""#p79300152"" class=""quotelink"">&gt;&gt;79300152</a><br>killer keemstar"
1,Readable text,killer keemstar
2,Explicit quote targets,"[('pol', 79300152)]"


## 2. Retrieve relevant threads

A whole thread is retained if at least one post matches a seed phrase. This preserves all its available replies, including posts without seeds and posts without text. By default, only seed-matching posts become discovery targets; other posts remain available as conversation context.

Seed matching retrieves a topic, **not a hate label**. For example, `jewish` also occurs in neutral discussion and counterspeech. The editable list is an initial retrieval choice. Inspect its effects before fixing an experiment. Counts below count matching posts, not embeddings or individual mentions.

In [3]:
posts, quote_edges, scan_stats, seed_counts, anchor_counts = prepare_raiders(
    SOURCE, SEED_TERMS, anchor_terms, max_threads=MAX_THREADS, progress_every=1000,
)
if posts.empty:
    raise RuntimeError("No threads matched. Inspect the preview and seed list.")
posts["timestamp_utc"] = pd.to_datetime(posts["timestamp"], unit="s", utc=True)
display(pd.DataFrame(scan_stats.items(), columns=["measure", "count"]))
print("Retained post dates:", posts["timestamp_utc"].min(), "to", posts["timestamp_utc"].max())
display(posts[["post_id", "thread_id", "is_op", "seed_matches", "text"]].head(8))

Scanned 1,000 threads / 40,656 posts; retained 294 threads
Retained post dates: 2016-06-30 03:14:20+00:00 to 2016-07-01 00:29:07+00:00


,measure,count
0,threads_scanned,1000
1,posts_scanned,40656
2,posts_with_text_scanned,38884
3,threads_retained,294
4,posts_retained,30731
5,explicit_quote_links,26191


,post_id,thread_id,is_op,seed_matches,text
0,79290593,79290593,True,[],Consider the following for those of you that think Assad is the good guy or the lesser evil. The Syrian government is responsible for 94% of the civilian de...
1,79290672,79290593,False,[],"the issue is creating a power vacuum by destabilizing a regime, no matter how ""evil"" creates worse problems than just waiting for its leader to die."
2,79290809,79290593,False,[],>Syrian Network for human rights\n>reliable source\n\ntry harder. Leaf.
3,79290815,79290593,False,[],If Assad is killing that many Arabs he is the good guy in my book
4,79290894,79290593,False,[],The current problems in Syria is due to the overhandedness of the regime in subduing rebellion. ISIS is ISIS because of the regime.
5,79290965,79290593,False,[],Sure lol
6,79290978,79290593,False,[],Actually it's a pretty reliable source considering the war is still going on. When the war ends the facts will more or less remain the same in who did the m...
7,79290992,79290593,False,[],We should probably stoke a civil war in Libya and remove Gaddafi while we're at it since he's a heavy-handed dictator. Surely that will bring peace and stab...


In [4]:
seed_table = pd.DataFrame({"seed": SEED_TERMS, "matching_posts_in_scan": [seed_counts.get(seed, 0) for seed in SEED_TERMS]})
display(seed_table.sort_values("matching_posts_in_scan", ascending=False))

thread_sizes = posts.groupby("thread_id").size()
display(thread_sizes.describe(percentiles=[0.5, 0.9, 0.99]).to_frame("posts_per_retained_thread"))

# Lightweight bars work without adding a plotting dependency.
def show_bars(series, title):
    maximum = max(float(series.max()), 1)
    rows = []
    for label, value in series.items():
        rows.append(f'<tr><td style="padding-right:12px">{html.escape(str(label))}</td><td style="width:300px"><div style="background:#16878b;height:14px;width:{100*value/maximum:.1f}%"></div></td><td>{int(value):,}</td></tr>')
    display(HTML(f"<b>{html.escape(title)}</b><table>{''.join(rows)}</table>"))

anchor_scan_table = pd.DataFrame({"anchor": list(anchor_terms), "matching_posts_in_scan": [anchor_counts.get(anchor, 0) for anchor in anchor_terms]})
show_bars(anchor_scan_table.set_index("anchor")["matching_posts_in_scan"].sort_values(ascending=False), "Anchor matches in all scanned posts")

,seed,matching_posts_in_scan
1,jews,634
0,jew,323
2,jewish,250
9,kike,160
4,zionist,64
13,holocaust,47
3,judaism,27
5,zionists,18
7,antisemitism,7
6,zionism,7


,posts_per_retained_thread
count,294.000000
mean,104.527211
std,120.450623
min,1.000000
50%,46.000000
90%,319.700000
99%,380.070000
max,421.000000


kike,,160
holocaust/gas chambers,,53
holohoax,,7
great replacement,,2
the goyim know,,1
blood libel,,1
jew coup,,0
jew down,,0
jewish feature,,0
jewish figure,,0
jewish lighting,,0


## 3. Inspect explicit conversation links

An edge points from the quoting post to the referenced post. `target_present` means that target exists in our retained table; absent targets are kept as unresolved references. Same-thread edges are the simplest starting point for the later conversation experiment. Quote links establish a reference, not agreement or endorsement.

In [5]:
edge_summary = pd.Series({
    "explicit quote links": len(quote_edges),
    "target present in retained posts": int(quote_edges["target_present"].sum()),
    "same-thread links": int(quote_edges["same_thread"].sum()),
    "cross-board links": int(quote_edges["target_board"].ne("pol").sum()),
})
display(edge_summary.to_frame("count"))
post_text = posts.set_index("post_id")["text"]
edge_examples = quote_edges.loc[quote_edges["same_thread"]].head(6).copy()
edge_examples["quoting_post_text"] = edge_examples["source_post_id"].map(post_text)
edge_examples["referenced_post_text"] = edge_examples["target_post_id"].map(post_text)
display(edge_examples[["source_post_id", "target_post_id", "quoting_post_text", "referenced_post_text"]])

,count
explicit quote links,26191
target present in retained posts,26149
same-thread links,26051
cross-board links,4


,source_post_id,target_post_id,quoting_post_text,referenced_post_text
0,79290672,79290593,"the issue is creating a power vacuum by destabilizing a regime, no matter how ""evil"" creates worse problems than just waiting for its leader to die.",Consider the following for those of you that think Assad is the good guy or the lesser evil. The Syrian government is responsible for 94% of the civilian de...
1,79290815,79290593,If Assad is killing that many Arabs he is the good guy in my book,Consider the following for those of you that think Assad is the good guy or the lesser evil. The Syrian government is responsible for 94% of the civilian de...
2,79290894,79290672,The current problems in Syria is due to the overhandedness of the regime in subduing rebellion. ISIS is ISIS because of the regime.,"the issue is creating a power vacuum by destabilizing a regime, no matter how ""evil"" creates worse problems than just waiting for its leader to die."
3,79290965,79290593,Sure lol,Consider the following for those of you that think Assad is the good guy or the lesser evil. The Syrian government is responsible for 94% of the civilian de...
4,79290978,79290809,Actually it's a pretty reliable source considering the war is still going on. When the war ends the facts will more or less remain the same in who did the m...,>Syrian Network for human rights\n>reliable source\n\ntry harder. Leaf.
5,79291033,79290815,I never thought about it that way. Thanks greatest ally!\n\nWell the main problem I guess is that muslims compete to see who is most muslim rather than just...,If Assad is killing that many Arabs he is the good guy in my book


## 4. Split whole threads and inspect duplicate text

The initial proportions are 20% reference, 20% development and 60% evaluation **by thread**. A stable hash of the thread ID assigns the role, so rescanning more threads does not reshuffle existing assignments. Fractions are approximate; post counts may be uneven.

Reference posts provide anchor contexts. Development posts guide choices. Evaluation posts are reserved for the final comparison; they do not yet have gold labels. This is an experimental split, not a BERT training split.

We keep complete context tables. For model inputs, equal cleaned text (ignoring case/whitespace) is deduplicated within each role. Reference posts copied anywhere in development/evaluation context are excluded. Development targets copied anywhere in evaluation context are excluded. This controls exact-text duplication, not near copies or quoted fragments. Preserved context can still contain repeated text, so any future context-based method needs its own leakage audit.

In [6]:
posts["split"] = posts["thread_id"].map(lambda thread_id: thread_role(
    thread_id, SPLIT_SALT, REFERENCE_FRACTION, DEVELOPMENT_FRACTION,
))
role_lookup = posts.set_index("post_id")["split"]
quote_edges["source_split"] = quote_edges["source_post_id"].map(role_lookup)
quote_edges["target_split"] = [role_lookup.get(target) if board == "pol" else None for board, target in zip(quote_edges["target_board"], quote_edges["target_post_id"])]
quote_edges["same_split"] = quote_edges["target_present"] & quote_edges["source_split"].eq(quote_edges["target_split"])
# Keep cross-split references in the audit table, but exclude them from context inputs.
context_edges = quote_edges.loc[quote_edges["same_thread"] & quote_edges["same_split"]].copy()

reference_posts, development_posts, evaluation_posts, excluded_reference_posts = select_model_inputs(posts, DISCOVERY_SCOPE)
role_summary = posts.groupby("split").agg(threads=("thread_id", "nunique"), context_posts=("post_id", "size"))
role_summary["model_input_posts"] = pd.Series({"reference": len(reference_posts), "development": len(development_posts), "evaluation": len(evaluation_posts)})
display(role_summary)
print("Duplicate nonempty text rows in retained contexts:", int(posts.loc[posts["text_hash"].notna(), "text_hash"].duplicated().sum()))
print("Reference rows excluded for overlap with discovery contexts:", len(excluded_reference_posts))
show_bars(role_summary["model_input_posts"], "Usable model inputs by role")

Duplicate nonempty text rows in retained contexts: 665
Reference rows excluded for overlap with discovery contexts: 0


,threads,context_posts,model_input_posts
split,,,
development,58,4328,186
evaluation,161,18340,756
reference,75,8063,77


development,,186
evaluation,,756
reference,,77


## 5. Check reference coverage and dataset integrity

An anchor match is a lexical observation, not proof of hateful meaning. Inspect reference examples before treating them as ground truth. Zero matches may reflect spelling, the limited slice, or the historical period. Coverage also needs checking after preprocessing and embedding.

The runner will need at least five eligible reference embeddings per candidate. The check below is only an early count of potential anchor/post pairs.

In [7]:
def coverage(frame):
    counts = Counter(anchor for matches in frame["anchor_matches"] for anchor in matches)
    return pd.Series({anchor: counts.get(anchor, 0) for anchor in anchor_terms})

coverage_table = pd.DataFrame({
    "all_scanned_posts": [anchor_counts.get(anchor, 0) for anchor in anchor_terms],
    "reference_posts": coverage(reference_posts),
    "development_targets": coverage(development_posts),
    "evaluation_targets": coverage(evaluation_posts),
}, index=list(anchor_terms))
coverage_table.index.name = "anchor"
display(coverage_table)
print("Reference anchors represented:", int(coverage_table["reference_posts"].gt(0).sum()), "/", len(anchor_terms))
print("Potential anchor/post pairs:", int(coverage_table["reference_posts"].sum()))
display(reference_posts[["post_id", "anchor_matches", "text"]].head(6))

assert posts["post_id"].is_unique
assert posts.groupby("thread_id")["split"].nunique().eq(1).all()
assert set(reference_posts["text_hash"]).isdisjoint(set(posts.loc[posts["split"].ne("reference"), "text_hash"]))
assert set(development_posts["text_hash"]).isdisjoint(set(posts.loc[posts["split"].eq("evaluation"), "text_hash"]))
for frame in (reference_posts, development_posts, evaluation_posts):
    assert frame["text_hash"].is_unique
    assert frame["Post Text"].str.len().gt(0).all()
assert context_edges["same_thread"].all() and context_edges["same_split"].all()
assert not reference_posts.empty and not development_posts.empty and not evaluation_posts.empty, "A role has no model inputs; expand the scan or review configuration."
assert coverage_table["reference_posts"].sum() >= 5, "Too few potential reference vectors."
print("Structural checks passed. Context labels still require human review.")

Reference anchors represented: 5 / 20
Potential anchor/post pairs: 77
Structural checks passed. Context labels still require human review.


,all_scanned_posts,reference_posts,development_targets,evaluation_targets
anchor,,,,
kike,160,56,14,90
blood libel,1,0,0,1
the goyim know,1,1,0,0
great replacement,2,1,0,1
holocaust/gas chambers,53,17,7,29
holohoax,7,2,2,3
jew coup,0,0,0,0
jew down,0,0,0,0
jewish feature,0,0,0,0


,post_id,anchor_matches,text
3818,79275801,[kike],The la times is owned and run by kikes you insane kike loving wacko
3828,79276065,[kike],It was probably revenge for Israel killing his little sister or something. Either way nothing of value was lost in both events. Kike's aern't human
3844,79276537,[kike],"Shut up kike sandnigger\n\nIn the 5th Century BCE, Herodotus, the first historian in Western civilization (the “father of history”) referenced ""Palestine"" n..."
3870,79278464,[kike],Typical kike always trying to shift the blame to White people. Why are you so obsessed Chaim?
3884,79279197,[kike],>Beautiful Jewess\n\nKike women are hideous. They look like inbred rats. That woman you posted is a Slav not a kike sandnigger
3894,79279876,[holocaust/gas chambers],If nobody cares about dead sandniggers why is the media constantly talking about the so called Holocaust?\n\ncheckm8


## 6. Export a versioned preparation folder

Every export creates a new folder under `data/raiders/prepared/`. `context_posts.jsonl` retains raw HTML, cleaned text, IDs, timestamps, matches and roles. `quote_edges.csv` retains all explicit references; `context_edges.csv` contains only resolved, same-thread/same-role links. Three CSVs contain model inputs with a `Post Text` column and stable `post_id` values.

The manifest records source identity, seeds, split settings, counts and helper-code checksum. A preview export is labelled as a preview. Existing files are not overwritten.

In [8]:
if EXPORT:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    scope = f"preview_{MAX_THREADS}_threads" if MAX_THREADS is not None else "full_1GB_slice"
    OUTPUT = ROOT / "data/raiders/prepared" / f"{scope}_{stamp}"
    OUTPUT.mkdir(parents=True, exist_ok=False)
    posts.to_json(OUTPUT / "context_posts.jsonl", orient="records", lines=True, force_ascii=False, date_format="iso")
    quote_edges.to_csv(OUTPUT / "quote_edges.csv", index=False)
    context_edges.to_csv(OUTPUT / "context_edges.csv", index=False)
    model_columns = ["post_id", "thread_id", "timestamp", "split", "Post Text", "text_hash"]
    for name, frame in [("reference_posts", reference_posts), ("development_posts", development_posts), ("evaluation_posts", evaluation_posts)]:
        frame[model_columns].to_csv(OUTPUT / f"{name}.csv", index=False)
    coverage_table.to_csv(OUTPUT / "anchor_coverage.csv")
    seed_table.to_csv(OUTPUT / "seed_coverage.csv", index=False)
    role_summary.to_csv(OUTPUT / "split_summary.csv")
    excluded_reference_posts[["post_id", "thread_id", "text_hash"]].to_csv(OUTPUT / "excluded_reference_duplicates.csv", index=False)
    source_info_path = SOURCE.with_suffix(".info.json")
    source_info = json.loads(source_info_path.read_text()) if source_info_path.exists() else None
    git = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, cwd=ROOT)
    manifest = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(), "mode": scope,
        "source": str(SOURCE.relative_to(ROOT)), "source_bytes": SOURCE.stat().st_size,
        "source_mtime_ns": SOURCE.stat().st_mtime_ns, "source_extraction_info": source_info,
        "max_threads": MAX_THREADS, "seed_terms": SEED_TERMS, "anchor_terms": anchor_terms,
        "split_salt": SPLIT_SALT, "reference_fraction": REFERENCE_FRACTION,
        "development_fraction": DEVELOPMENT_FRACTION, "discovery_scope": DISCOVERY_SCOPE,
        "scan_statistics": scan_stats, "split_counts": role_summary.to_dict(orient="index"),
        "git_commit": git.stdout.strip() if git.returncode == 0 else None,
        "preparation_helper_sha256": hashlib.sha256((ROOT / "utils/raiders.py").read_bytes()).hexdigest(),
        "limitations": ["First threads in archive order; no representative sampling", "Seed matches are not hate annotations", "No near-duplicate removal", "Future context extensions require a repeated-context leakage audit"],
    }
    (OUTPUT / "preparation_info.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    display(pd.DataFrame([{"file": path.name, "MB": round(path.stat().st_size / 1e6, 3)} for path in sorted(OUTPUT.iterdir())]))
    print("Saved:", OUTPUT.relative_to(ROOT))
else:
    print("Export disabled. The prepared tables remain in memory.")

Saved: data/raiders/prepared/preview_1000_threads_20261025T203729_070670Z


,file,MB
0,anchor_coverage.csv,0.001
1,context_edges.csv,2.053
2,context_posts.jsonl,21.381
3,development_posts.csv,0.077
4,evaluation_posts.csv,0.249
5,excluded_reference_duplicates.csv,0.000
6,preparation_info.json,0.004
7,quote_edges.csv,2.063
8,reference_posts.csv,0.023
9,seed_coverage.csv,0.000


## Next checkpoint

Review the seed list, retained contexts, missing anchors and split sizes. When these choices are suitable, set `MAX_THREADS = None` and rerun to prepare the full 1 GB slice.

**Before BERT:** adapt the existing reproduction runner to accept the reference CSV separately from the discovery CSV and preserve `post_id`. Its current single-corpus input would rebuild anchors from discovery posts, bypassing the separation prepared here. Choose either development or evaluation targets explicitly; keep evaluation results out of tuning decisions. No runner changes are made by this notebook.